<a href="https://colab.research.google.com/github/lilianacervantes/Docs-HTML-CSS/blob/master/GuitarTabs.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Building a constraint system that forces music into easy guitar shapes

Core principle

For every note, you choose the easiest possible fret position, based on:

minimal finger movement
preference for open strings
low fret positions (1–5 max)
avoiding big jumps
staying on one or two strings when possible

Use standard tuning

scoring system: score = fret_cost + movement_cost + string_penalty
fret_cost = fret_number * 2 -- low frets preferred
movement_cost = abs(current_fret - candidate_fret) -- discourage big jumps
string_penalty =
    0 if open string
    1 if same string as previous note
    3 if switching strings
  --prefer open strings


In [202]:
!pip install pretty_midi

In [203]:
## TEST MIDI FILE
import pretty_midi

# ---------- CREATE MIDI OBJECT ----------
midi = pretty_midi.PrettyMIDI()
instrument = pretty_midi.Instrument(program=0)  # piano-like sound

# ---------- SIMPLE MELODY (Twinkle-style) ----------
melody = [
    ("C4", 0.0),
    ("C4", 0.5),
    ("G4", 1.0),
    ("G4", 1.5),
    ("A4", 2.0),
    ("A4", 2.5),
    ("G4", 3.0),
    ("REST", 3.5)
]

# ---------- ADD NOTES ----------
for note_name, start in melody:
    if note_name == "REST":
        continue

    note = pretty_midi.Note(
        velocity=100,
        pitch=pretty_midi.note_name_to_number(note_name),
        start=start,
        end=start + 0.4
    )
    instrument.notes.append(note)

midi.instruments.append(instrument)

# ---------- SAVE FILE ----------
midi.write("test_melody.mid")

print("MIDI file created: test_melody.mid")

MIDI file created: test_melody.mid


In [204]:

midi = pretty_midi.PrettyMIDI("test_melody.mid")

print("INSTRUMENTS:", len(midi.instruments))

for i, inst in enumerate(midi.instruments):
    print("Instrument", i, "notes:", len(inst.notes))

INSTRUMENTS: 1
Instrument 0 notes: 7


In [205]:
## Load the file
import pretty_midi

midi = pretty_midi.PrettyMIDI("test_melody.mid")
nnotes = []

for inst in midi.instruments:
    for note in inst.notes:
        notes.append((note.start, note.pitch))

In [206]:
## MIDI notes to guitar frets

def midi_to_name(pitch):
    return pretty_midi.note_number_to_name(pitch)

In [207]:
## Beginner guitar map -- fret positions
NOTE_MAP = {
    # Low octave
    "E3": [(6, 0)],
    "F3": [(6, 1)],
    "G3": [(6, 3)],
    "A3": [(5, 0), (6, 5)],
    "B3": [(5, 2)],
    "C4": [(5, 3)],
    "D4": [(4, 0), (5, 5)],

    # Main beginner zone (STRICTLY low fret focused)
    "E4": [(4, 2), (3, 2), (2, 5), (1, 0)],
    "F4": [(4, 3), (3, 3)],
    "G4": [(3, 0), (2, 0), (1, 3)],
    "A4": [(3, 2), (2, 2), (1, 5)],
    "B4": [(2, 0), (1, 2)],
    "C5": [(2, 1), (1, 3)],
    "D5": [(2, 3), (1, 5)],
    "E5": [(1, 0)],
}

In [208]:
## Logic to prefer beginner-friendly playing
def score(pos, prev):
    string, fret = pos

    # HARD RULE: keep it beginner-friendly
    if fret > 5:
        return 999

    fret_cost = fret * 2
    movement_cost = abs(fret - prev["fret"])
    string_penalty = 0 if string == prev["string"] else 2

    return fret_cost + movement_cost + string_penalty

In [209]:
## Choose best position per note
def choose_best(note, prev):
    if note not in NOTE_MAP:
        return None

    best = None
    best_score = 999

    for pos in NOTE_MAP[note]:
        s = score(pos, prev)
        if s < best_score:
            best_score = s
            best = pos

    return best

In [210]:
## Convert MIDI notes to guitar positions
melody_notes = [midi_to_name(pitch) for _, pitch in notes]

tab = {1: [], 2: [], 3: [], 4: [], 5: [], 6: []}

prev = {"string": 1, "fret": 0}

In [211]:
print("melody_notes OBJECT:", melody_notes)
print("TYPE:", type(melody_notes))
print("LENGTH:", len(melody_notes))

melody_notes OBJECT: ['C4', 'C4', 'G4', 'G4', 'A4', 'A4', 'G4', 'C4', 'C4', 'G4', 'G4', 'A4', 'A4', 'G4', 'C4', 'C4', 'G4', 'G4', 'A4', 'A4', 'G4', 'C4', 'C4', 'G4', 'G4', 'A4', 'A4', 'G4', 'C4', 'C4', 'G4', 'G4', 'A4', 'A4', 'G4', 'C4', 'C4', 'G4', 'G4', 'A4', 'A4', 'G4', 'C4', 'C4', 'G4', 'G4', 'A4', 'A4', 'G4', 'C4', 'C4', 'G4', 'G4', 'A4', 'A4', 'G4', 'C4', 'C4', 'G4', 'G4', 'A4', 'A4', 'G4', 'C4', 'C4', 'G4', 'G4', 'A4', 'A4', 'G4', 'C4', 'C4', 'G4', 'G4', 'A4', 'A4', 'G4']
TYPE: <class 'list'>
LENGTH: 77


In [212]:
## Build the tab

ttab = {1: [], 2: [], 3: [], 4: [], 5: [], 6: []}
step = 0

for note in melody_notes:
    # always create a full column first
    for s in tab:
        tab[s].append("-")

    pos = choose_best(note, prev)

    if pos:
        string, fret = pos
        tab[string][step] = str(fret)

    step += 1

In [214]:
## Print the results
strings = ["e", "B", "G", "D", "A", "E"]

# find columns that actually contain notes
keep = []
for i in range(len(tab[1])):
    if any(tab[s][i] != "-" for s in tab):
        keep.append(i)

# print compressed tab
for i, s in enumerate(strings, start=1):
    line = "".join(tab[i][j] for j in keep)
    print(f"{s}|{line}|")

e|-----------------------------------------------------------------------------|
B|-----------------------------------------------------------------------------|
G|--00220--00220--00220--00220--00220--00220--00220--00220--00220--00220--00220|
D|-----------------------------------------------------------------------------|
A|33-----33-----33-----33-----33-----33-----33-----33-----33-----33-----33-----|
E|-----------------------------------------------------------------------------|
